# 02c: physical-scale cache pilot

The baseline cache resizes every slice to 256 px whatever its field of view, so the knee
can be large in one study and small and off-centre in another. Geometry "mm" resamples
each series to a fixed 140 mm / 256 px (about 0.55 mm per pixel) and crops a 256 x 256
window centred on the leg. Output shape stays 32 x 256 x 256, so the model code does not
change; only the cache does.

This pilot compares both geometries on 10 studies by eye, then measures time and size of
the mm version on 40 studies before a full build.

Settings: CPU, Internet off.
Inputs: competition data, 02b output (selected_series.csv), utility script knee_prep.

# Cell 1: inputs

In [ ]:
import os, sys, glob, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT = Path("/kaggle/working")

def find_all(pattern):
    hits = glob.glob(f"/kaggle/usr/lib/**/{pattern}", recursive=True)
    for top in os.listdir("/kaggle/input"):
        if top != "competitions":
            hits += glob.glob(f"/kaggle/input/{top}/**/{pattern}", recursive=True)
    return sorted(hits)

kp_path = find_all("knee_prep.py")[0]
sys.path.insert(0, os.path.dirname(kp_path))
import knee_prep as kp
print(kp_path, "| mm geometry available:", hasattr(kp, "resample_mm"), "| MM_FOV", getattr(kp, "MM_FOV", None))

sel = pd.read_csv(find_all("selected_series.csv")[0])
ids = sel["StudyInstanceUID"].drop_duplicates().sample(10, random_state=7).tolist()
print(len(sel), "selected series;", len(ids), "pilot studies")

# Cell 2: px vs mm on 10 studies (mid slice of each plane)

In [ ]:
fig, axes = plt.subplots(len(ids), 6, figsize=(18, 3 * len(ids)))
rows_qa = []
for i, sid in enumerate(ids):
    rows = sel[sel["StudyInstanceUID"] == sid]
    out = {}
    for geo in ["px", "mm"]:
        t0 = time.time()
        vols, qas = kp.process_study(rows, ROOT, "train_series", geometry=geo)
        out[geo] = vols
        if geo == "mm":
            for q in qas:
                rows_qa.append({"study": sid[-6:], "plane": q["plane"], "px": q.get("px"), "n": q["n"],
                                "sec_mm": round(time.time() - t0, 2)})
    for j, key in enumerate(["sag", "cor", "axi"]):
        for g, geo in enumerate(["px", "mm"]):
            ax = axes[i, 2 * j + g]
            v = out[geo].get(key)
            if v is not None:
                ax.imshow(v[kp.D // 2], cmap="gray", vmin=0, vmax=255)
            ax.set_title(f"{sid[-6:]} {key} {geo}", fontsize=8)
            ax.axis("off")
plt.tight_layout()
plt.show()
qa = pd.DataFrame(rows_qa)
qa["fov_note"] = qa["px"].map(lambda p: "no spacing" if p is None else f"{p[0]:.2f} mm/px")
print(qa.to_string())

# Cell 3: time and size of the mm cache on 40 studies

In [ ]:
pilot = sel[sel["StudyInstanceUID"].isin(sel["StudyInstanceUID"].drop_duplicates().sample(40, random_state=11))]
t0 = time.time()
res = kp.build_cache(pilot, ROOT, "train_series", str(OUT / "pilot_mm"), workers=4, geometry="mm")
el = time.time() - t0
files = list((OUT / "pilot_mm").glob("*.npz"))
mb = np.array([f.stat().st_size / 1e6 for f in files])
print(f"{len(files)} studies in {el:.0f}s -> {el / len(files):.2f} s/study with 4 workers, "
      f"about {el / len(files) * 4407 / 3600:.2f} h for train")
print(f"{mb.mean():.2f} MB/study (max {mb.max():.2f}) -> about {mb.mean() * 4407 / 1e3:.1f} GB for train "
      f"(baseline cache: 3.82 MB/study, 16.8 GB)")
print("errors:", int(res["error"].notna().sum()) if len(res) else 0)

# Cell 4:

In [ ]:
import torch
import torch.nn.functional as F

def coverage(sid):
    out = []
    for r in sel[sel["StudyInstanceUID"] == sid].itertuples(index=False):
        vol, qa = kp.load_volume(os.path.join(ROOT, "train_series", r.StudyInstanceUID, r.SeriesInstanceUID))
        if vol is None or not qa.get("px"):
            continue
        v = kp.normalize(vol)
        t_mm = kp.MM_FOV / kp.S
        nh = max(1, round(v.shape[1] * qa["px"][0] / t_mm))
        nw = max(1, round(v.shape[2] * qa["px"][1] / t_mm))
        img = F.interpolate(torch.from_numpy(v.mean(0, keepdims=True))[None], size=(nh, nw), mode="bilinear",
                            antialias=True, align_corners=False)[0, 0].numpy()
        m = kp.leg_mask(img)
        _, (cy, cx) = kp.knee_crop(img[None], kp.S)
        y0, x0 = cy - kp.S // 2, cx - kp.S // 2
        ry0, ry1, rx0, rx1 = max(y0, 0), y0 + kp.S, max(x0, 0), x0 + kp.S
        in_rows = m[ry0:ry1].sum()
        out.append({"study": sid[-6:], "plane": r.Anatomical_Plane[:3],
                    "fov_mm": round(max(vol.shape[1] * qa["px"][0], vol.shape[2] * qa["px"][1])),
                    "kept_all": m[ry0:ry1, rx0:rx1].sum() / max(m.sum(), 1),
                    "kept_width": m[ry0:ry1, rx0:rx1].sum() / max(in_rows, 1)})
    return out

cov = pd.DataFrame([row for sid in pilot["StudyInstanceUID"].unique() for row in coverage(sid)])
cov["key"] = np.where(cov["plane"] == "Axi", cov["kept_all"], cov["kept_width"])
print(cov.groupby("plane")[["kept_all", "kept_width"]].median().round(3).to_string())
print(cov.groupby("plane")["key"].describe(percentiles=[0.05, 0.25, 0.5]).round(3).to_string())
print("\nseries with key < 0.9:", int((cov["key"] < 0.9).sum()), "of", len(cov))
print(cov.sort_values("key").head(8).round(3).to_string())